**STEP 1: Simulation**

In [1]:
import numpy as np
import random
from scipy.stats import wishart
def simulate_matrix(n, d, no_components,rho,abs_bound_X):
    ###Form the X matrix
    X = np.zeros((0, d))
    # Use base_value and reminder to evenly distributed the number of samples in each component
    def generate_components(n, no_components):
        base_value = n // no_components
        remainder = n % no_components
        components = [base_value] * no_components
        for i in range(remainder):
            components[i] += 1
        random.shuffle(components)
        if sum(components) > n:
            components[n-1] -= sum(components) - n
        elif sum(components) < n:
            components[n-1] += n - sum(components)
        return components
    # Generate these multivariate gaussian components in different levels of mean and cov.
    num_list = generate_components(n, no_components)
    for i in range(no_components):
        # each level's mean value, grow by 4's power.
        mean_value = 5**(i+1)
        # 0.8 probability of 1 and 0.2 probability of 0 for creating a sparse mean vector.
        mean_vector = np.random.binomial(1, 0.8, size = d)
        mean = mean_value * mean_vector
        # randomly generate a cov matrix and make sure that it is symmetric.
        cov = wishart.rvs(df=d, scale=np.eye(d))
        # add a small value to the diagonal to ensure positive-definiteness
        cov += np.eye(d) * 1e-6
        # generate a component of X.
        X_ = np.random.multivariate_normal(mean, cov, num_list[i])
        X_ = np.array(X_)
        # append the component to the list.
        X = np.vstack((X, X_))
    X = np.array(X)
    # Set the elements < 3 to 0 to make it a sparse matrix
    X[abs(X) < abs_bound_X] = 0
    ###Form the fixed beta
    np.random.seed(39)
    beta_value = np.random.binomial(1, 0.8, size = d)
    beta = (beta_value * np.random.uniform(-1, 1, size = d)).reshape(-1,1)
    beta = np.array(beta)
    np.random.seed(None)
    ###Form the error term - epsilon, which has the auto-correlation structure in time series analysis.
    r = rho # autocorrelation coefficient
    epsilon_matrix = np.zeros((0,1)) # define a zero initial matrix for vstack
    ###Form the error term in the same levels
    for i in range(no_components):
        epsilon = np.zeros((num_list[i],1))
        epsilon[0] = np.random.normal((5**(i+1))/100,1)
        for t in range(1,num_list[i]):
            epsilon[t] = r * epsilon[t-1] + np.random.normal((5**(i+1))/100,1)
        epsilon = np.array(epsilon)
        epsilon_matrix = np.vstack((epsilon_matrix, epsilon))
    epsilon_matrix = np.array(epsilon_matrix)
    epsilon = epsilon_matrix
    ###Form the Y matrix
    Y = np.array(X @ beta + epsilon)
    ###Count the number of elements < 0.1 in X and Y
    return X,Y,beta,epsilon

# Matrix Settled
n = 10**4       # number of rows         
d = 500        # number of columns
no_components = 4 # number of levels of multivariate gaussian distribution in X
rho = 0.7 # autocorrelation coefficient in time series analysis
abs_bound_X = 2 # absolute bound for X to set elements to be 0
X,Y,beta,epsilon = simulate_matrix(n, d, no_components,rho,abs_bound_X)
print("The simulated matrix X is:")
print(X)
print("The simulated vector Y is:")
print(Y)

The simulated matrix X is:
[[ 10.96084429 -15.7487948    5.1426583  ...  28.8230022   73.09072929
   -5.79676743]
 [-37.92295745  36.64615389  22.1224294  ... -20.41984555 -11.67230194
  -15.41940906]
 [-16.66017125  23.09509762  -4.96578148 ...  14.5748886    4.2397587
    5.09207937]
 ...
 [623.08935322  12.82798166 603.84504684 ... 627.27519056 636.34188145
  636.64673361]
 [633.90686574  -8.39357041 600.01450383 ... 633.74659    584.0341709
  613.53363495]
 [649.92360738  37.39890288 652.37159688 ... 648.32111156 591.7344008
  635.42775283]]
The simulated vector Y is:
[[-657.1331272 ]
 [ 355.87943142]
 [-280.8059089 ]
 ...
 [1261.97362385]
 [1148.89243356]
 [1551.99749024]]


*Define functions for calculating ratio and norms*

In [2]:
def calculate_the_norm_square(A,b,x_selected):
    return (np.linalg.norm(A @ x_selected - b)) ** 2
def abs_error_ratio(A,b,x_hat_bar,x_star):
    return calculate_the_norm_square(A,b,x_hat_bar) - calculate_the_norm_square(A,b,x_star) / calculate_the_norm_square(A,b,x_star)

*Try to figure out what x_star should be exiquisitely*

In [3]:
from scipy.linalg import cho_factor, cho_solve
import time
def solver(A,b,solver):
    # cholesky decomposition algorithm
    if solver == 'cholesky':
        L,low = cho_factor(A.T@A)
        x_star = cho_solve((L,low),A.T@b)
    # Direct solver
    elif solver == 'direct':
        x_star = np.linalg.inv(A.T@A)@A.T@b
    return x_star
start_cholesky = time.time()
x_star_cholesky = solver(X,Y,'cholesky')
end_cholesky = time.time()
execution_cholesky = end_cholesky - start_cholesky
start_direct = time.time()
x_star_direct = solver(X,Y,'direct')
end_direct = time.time()
execution_direct = end_direct - start_direct
print("The execution time of cholesky decomposition is:")
print(execution_cholesky)
print("The execution time of direct solver is:")
print(execution_direct)
norm_cholesky = calculate_the_norm_square(X,Y,x_star_cholesky)
norm_direct = calculate_the_norm_square(X,Y,x_star_direct)
print("The norm square of x_star calculated as by cholesky and direct solver respectively are:")
print(norm_cholesky, norm_direct)
minimum_norm_square = min(norm_cholesky, norm_direct)
print("Here we output the minimum of the norm suqare for x_star calculated as:")
print(minimum_norm_square)
print("Their difference by cholesky minus direct solver is:")
print(norm_cholesky - norm_direct)

The execution time of cholesky decomposition is:
0.018579959869384766
The execution time of direct solver is:
0.15433955192565918
The norm square of x_star calculated as by cholesky and direct solver respectively are:
19646.1179081056 19646.117908105673
Here we output the minimum of the norm suqare for x_star calculated as:
19646.1179081056
Their difference by cholesky minus direct solver is:
-7.275957614183426e-11


###It is obvious that norm difference of cholsky and direct solver is very small.

**Consider the number of rows (n) of the matrix X and Y in the scale of powers of 10.**<br>
*In my computation environment, when n = 10^6, the computation time of simulation is only 15.2s in one test case.*<br>
*X_star by cholesky takes 0.939s and x_star by direct solver takes 2.464s*<br>
*But when n = 10^7, the computation time of simulation is as large as 21m 9.5s in one test case.*<br>
*X_star by cholesky and x_star by direct solver even collapse in time and memory.*

In [4]:
#Simulation time when n = 10^6
from IPython.display import Markdown
Markdown('![Image Description](./10^6_simulation.png)')

![Image Description](./10^6_simulation.png)

In [5]:
#Cholesky vs direct solver time when n = 10^6
Markdown('![Image Description](./10^6_two_times.png)')

![Image Description](./10^6_two_times.png)

In [6]:
#Simulation time when n = 10^7
Markdown('![Image Description](./10^7_simulation.png)')

![Image Description](./10^7_simulation.png)

In [7]:
###From the above different norm and optimal solver calculations, it is obvious that:
###the direct solver and cholesky decomposition are the optimal solvers.
###Since their norms are very very very nearly the same
###we choose the direct solver as our optimal solver when matrix is not so large.
###And we choose the cholesky decomposition as our optimal solver when matrix is so large.
# And the judge of line is at n = 10^6
if n <= 10**6:
    x_Star = x_star_direct
    norm_Star = norm_direct
else:
    x_Star = x_star_cholesky
    norm_Star = norm_cholesky
print("The optimal solver is:")
print(x_Star)

The optimal solver is:
[[-1.16039344e-01]
 [ 7.46370742e-01]
 [-1.05866266e-03]
 [-4.42311563e-01]
 [ 9.55433635e-01]
 [ 3.74687774e-01]
 [-1.35632134e-01]
 [-9.00129117e-01]
 [ 9.56740978e-02]
 [-8.70897185e-04]
 [ 7.23879079e-04]
 [ 5.05370335e-04]
 [-6.91353754e-04]
 [-1.69511015e-01]
 [-3.71386992e-04]
 [-9.76022169e-04]
 [-5.99456924e-01]
 [ 8.66989368e-01]
 [-4.58118502e-01]
 [-2.73242492e-04]
 [ 4.10890153e-01]
 [ 9.01397288e-04]
 [-4.84687249e-01]
 [-4.07663631e-01]
 [ 9.10430489e-02]
 [-4.74968057e-04]
 [-1.71022499e-01]
 [ 1.06842070e-01]
 [-2.58564862e-01]
 [-3.18113241e-01]
 [-1.16847982e-01]
 [ 1.38039189e-01]
 [-9.52752179e-02]
 [ 5.83285657e-04]
 [ 5.99636511e-01]
 [ 4.96432831e-01]
 [ 9.64488663e-01]
 [-2.35955699e-04]
 [-6.06602179e-01]
 [-2.50646414e-04]
 [-6.21674993e-01]
 [ 3.97455856e-01]
 [ 2.07458357e-01]
 [-1.91096037e-04]
 [-6.26863155e-01]
 [-1.16242142e-01]
 [ 2.16916947e-01]
 [-6.76056076e-01]
 [-2.01225715e-01]
 [-9.22744287e-01]
 [ 8.34379817e-01]
 [ 8.845

**STEP 2: Uniform Sampling sketch // Algorithm 1: Distributed Randomized Regression**

In [8]:
# e.g. our desired sketching size is m = 1000
from operator import index


m = 1000

# Algorithm 1 inserting inside uniform sampling: Distributed Randomized Regression

# S_k @ A here is just computed as A [uniformly_sampled_index]
# As S_k here is just a diagnoal matrix of 1 or 0 where sampled rows have 1 as value
def uniform_sampling(A,b,n,m,q):
    x_hat_list = []
    for i in range(q):
        index = np.random.choice(n, size=m, replace=False)
        A_sk = A[index]
        b_sk = b[index]
        x_hat = np.linalg.inv(A_sk.T @ A_sk) @ A_sk.T @ b_sk
        x_hat_list.append(x_hat)
    x_bar = sum(x_hat_list) / q
    return x_bar
